# 01 – Exploring the raw TTC delay data

Goal: look at every raw file **before** writing any cleaning code, and write down every inconsistency.
Nothing here changes the data. The findings are collected in [`../DATA_NOTES.md`](../DATA_NOTES.md),
which becomes the to-do list for `normalize.py` and the `clean_*.py` scripts.

Sections: **1. Setup · 2. Subway · 3. Bus · 4. Streetcar · 5. Duplicate files · 6. Summary**

Loading every Excel sheet takes about 2 minutes (the bus files are big).

## 1. Setup

In [1]:
import hashlib
from collections import defaultdict
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")   # the notebook lives in notebooks/, data is one level up

In [2]:
def load_sheets(mode):
    """Read every delay file for one mode ("subway", "bus", "streetcar").

    Returns a list of DataFrames, one per Excel sheet (or one for the CSV),
    each tagged with the file and sheet it came from.
    """
    parts = []
    for f in sorted((RAW / f"ttc-{mode}-delay-data").glob("*")):
        if "readme" in f.name or "code" in f.name:
            continue                                   # not delay records
        if f.suffix == ".csv":
            sheets = {"csv": pd.read_csv(f)}
        else:
            sheets = pd.read_excel(f, sheet_name=None)  # None = ALL sheets
        for sheet_name, df in sheets.items():
            parts.append(df.assign(_file=f.name, _sheet=sheet_name))
    return parts


def column_layouts(parts):
    """Group sheets by their exact column names, to see how the layout changes over time."""
    groups = defaultdict(list)
    for df in parts:
        cols = tuple(c for c in df.columns if c not in ("_file", "_sheet"))
        groups[cols].append(f"{df['_file'].iat[0]} [{df['_sheet'].iat[0]}]")
    for cols, where in groups.items():
        print(f"{len(where):>3} sheet(s): {list(cols)}")
        print(f"      first: {where[0]}")
        print(f"      last:  {where[-1]}\n")


# Exploration-only rename so the three datasets can be compared side by side.
# The real, tested version of this mapping goes in the clean_*.py scripts.
RENAME = {
    "Report Date": "date", "Date": "date",
    "Route": "route", "Line": "route",
    "Time": "time", "Day": "day",
    "Location": "location", "Station": "location",
    "Incident": "incident", "Code": "incident",
    "Min Delay": "min_delay", " Min Delay": "min_delay", "Delay": "min_delay",
    "Min Gap": "min_gap", "Gap": "min_gap",
    "Direction": "bound", "Bound": "bound",
    "Vehicle": "vehicle",
}

## 2. Subway

In [3]:
subway_parts = load_sheets("subway")
column_layouts(subway_parts)

 60 sheet(s): ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
      first: ttc-subway-delay-2020.xlsx [Jan 2020]
      last:  ttc-subway-delay-may-december-2017.xlsx [Dec17]

  1 sheet(s): ['_id', 'Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
      first: ttc-subway-delay-data-since-2025.csv [csv]
      last:  ttc-subway-delay-data-since-2025.csv [csv]



**Finding:** subway is the tidy one. Every sheet has the same 10 columns; the since-2025 CSV only adds `_id`.
Older files split the year into monthly sheets (with inconsistent sheet names); 2022+ have one sheet per file.

In [4]:
subway = pd.concat(subway_parts, ignore_index=True)
print(len(subway), "rows")
subway.groupby("_file")["Date"].agg(["min", "max", "count"])

267842 rows


,min,max,count
_file,,,
ttc-subway-delay-2020.xlsx,2020-01-01 00:00:00,2020-12-31 00:00:00,14782
ttc-subway-delay-2021.xlsx,2021-01-01 00:00:00,2021-12-31 00:00:00,16370
ttc-subway-delay-2022.xlsx,2022-01-01 00:00:00,2022-12-31 00:00:00,19895
ttc-subway-delay-2023.xlsx,2023-01-01 00:00:00,2023-12-31 00:00:00,22949
ttc-subway-delay-2024.xlsx,2024-01-01 00:00:00,2024-12-31 00:00:00,26467
ttc-subway-delay-data-2018.xlsx,2018-01-01 00:00:00,2018-12-31 00:00:00,20737
ttc-subway-delay-data-2019.xlsx,2019-01-01 00:00:00,2019-12-31 00:00:00,19222
ttc-subway-delay-data-since-2025.csv,2025-01-01,2026-08-31,45475
ttc-subway-delay-jan-2014-april-2017.xlsx,2014-01-01 00:00:00,2017-04-30 00:00:00,69016


Date ranges line up with the file names, with no gaps or overlaps. `Date` is a real datetime in the Excel files but a string in the CSV.

In [5]:
print(subway.dtypes, "\n")
subway.isna().sum()

Date          object
Time             str
Day              str
Station          str
Code             str
Min Delay      int64
Min Gap        int64
Bound            str
Line             str
Vehicle        int64
_file            str
_sheet           str
_id          float64
dtype: object 



Date              0
Time              0
Day               0
Station           0
Code              1
Min Delay         0
Min Gap           0
Bound         74002
Line            876
Vehicle           0
_file             0
_sheet            0
_id          222367
dtype: int64

In [6]:
# Line: should be 4 values (YU, BD, SHP, SRT)
print(subway["Line"].nunique(), "distinct values")
subway["Line"].value_counts(dropna=False)

111 distinct values


Line
YU                   131035
BD                   114837
SHP                    9495
SRT                    7672
YU/BD                  3360
                      ...  
32 EGLINTON WEST          1
129 MCCOWAN NORTH         1
YU BD                     1
YU - BD LINE              1
85 SHEPPARD EAST          1
Name: count, Length: 112, dtype: int64

**Finding:** `Line` has 111 spellings for 4 lines: `YUS`, `B/D`, `BD LINE`, `LINE 1`, `BLOOR DANFORTH`…,
combined lines (`YU/BD`, `YU / BD`, `BD/YU`, `YU & BD`…), bus routes that don't belong here (`29 DUFFERIN`, `504 KING`),
junk (`999`, `TRANSIT CONTROL`), and 876 blanks.

In [7]:
# Station: Toronto has ~75 subway stations
print(subway["Station"].nunique(), "distinct values")
print("longest name:", subway["Station"].str.len().max(), "characters")
subway["Station"].value_counts().head(40)

2223 distinct values
longest name: 22 characters


Station
KENNEDY BD STATION       10414
FINCH STATION            10055
KIPLING STATION           9980
BLOOR STATION             8131
EGLINTON STATION          6994
WARDEN STATION            5832
WILSON STATION            5697
SHEPPARD WEST STATION     5307
VICTORIA PARK STATION     5031
YONGE BD STATION          4926
KEELE STATION             4789
ISLINGTON STATION         4656
COXWELL STATION           4386
VAUGHAN MC STATION        4371
UNION STATION             4365
DUNDAS STATION            4359
YONGE UNIVERSITY LINE     4258
ST GEORGE YUS STATION     4258
DAVISVILLE STATION        4035
ST CLAIR WEST STATION     3947
ST GEORGE BD STATION      3824
YORK MILLS STATION        3606
LAWRENCE STATION          3533
ST CLAIR STATION          3527
WELLESLEY STATION         3397
COLLEGE STATION           3397
SHEPPARD STATION          3357
DUNDAS WEST STATION       3262
JANE STATION              3217
GREENWOOD STATION         3200
BROADVIEW STATION         3147
QUEEN STATION             3091


In [8]:
# The long tail: typos, truncation, notes, and places that aren't stations
subway["Station"].value_counts().tail(40)

Station
COXWELL STATION (EXITI    1
MCCOWAN STATION (ENTER    1
MUSEUM STATION (LEAVIN    1
ELLESMERE STATION DEPA    1
DUNDAS STATION - WEST     1
YONGE-UNIVERSITY SUBW     1
ISLINGTON STATION (EXI    1
UNION STATION (APPROAC    1
BROADVIEW TO SHERBOURN    1
BROADVIEW TO YONGE STA    1
169 DANFORTH AVENUE       1
WARDEN STATION (APPR      1
DUPONT STATION ( APPRO    1
ST CLAIR STATION PLEAS    1
LESLIE STATION APROACH    1
UNION STATION - SALES     1
KENNEDY BD STATION ( A    1
DUPONT STATION APPROAC    1
WILSON STATION (EXITIN    1
DOWNSVIEWSTATION          1
YONGE/UNIVERSITY BLOOR    1
YONGE/UNIVERISITY-BLOO    1
YONGE/UNIVERSITY & BLO    1
YONGE/UINVERSITY & BLO    1
WELLESLEY TO COLLEGE      1
SHEPPARD TO BAYVIEW       1
BLOOR STATON              1
VICTORIA PARK             1
MC BRIEN BUILDING         1
EGLINTON WEST TO ST CL    1
YORK MILLS STATION (A     1
WILSON STRUCTURE BUILD    1
ATC TERRIROTY             1
YORKDALE TO DUPONT STA    1
CLANTON PARK PORTAL       1
DANFORTH AND

**Finding:** 2,223 distinct station strings for ~75 stations.
- Names are **cut off at 22 characters** (`PIONEER VILLAGE STATIO`, `YONGE-UNIVERSITY AND B`).
- Interchange stations are split by line: `ST GEORGE YUS STATION` / `ST GEORGE BD STATION`, `KENNEDY BD` / `KENNEDY SRT`.
- Aliases: `VMC STATION` vs `VAUGHAN MC STATION`, `BLOOR STATION` vs `YONGE BD STATION`.
- Not stations: whole lines (`YONGE UNIVERSITY LINE`, `BLOOR DANFORTH SUBWAY`), segments (`WELLESLEY TO COLLEGE`), buildings/yards (`WILSON CAPITAL SHOP`).
- Typos and notes: `BLOOR STATON`, `DOWNSVIEWSTATION`, `WILSON STATION (EXITIN`.

In [9]:
print(subway["Bound"].value_counts(dropna=False), "\n")
print(subway["Day"].value_counts())

Bound
NaN    74002
S      54657
W      47016
N      46530
E      45253
B        379
Y          2
0          1
5          1
R          1
Name: count, dtype: int64 

Day
Thursday     41689
Wednesday    41466
Friday       41286
Tuesday      41125
Monday       40518
Saturday     33019
Sunday       28739
Name: count, dtype: int64


`Bound` should be N/S/E/W; it also has `B`, `Y`, `R`, `0`, `5` and 74k blanks. `Day` is clean.

In [10]:
print(subway["Min Delay"].describe(), "\n")
print("share with 0 min delay:", round((subway["Min Delay"] == 0).mean(), 3))
print("largest delays:", sorted(subway["Min Delay"])[-8:])
print("share with Vehicle = 0:", round((subway["Vehicle"] == 0).mean(), 3))
print("exact duplicate rows:", subway.drop(columns=["_file", "_sheet", "_id"]).duplicated().sum())

count    267842.000000
mean          2.552833
std          10.008942
min           0.000000
25%           0.000000
50%           0.000000
75%           3.000000
max         999.000000
Name: Min Delay, dtype: float64 

share with 0 min delay: 0.652
largest delays: [661, 708, 716, 788, 807, 827, 900, 999]
share with Vehicle = 0: 0.328


exact duplicate rows: 269


**Finding:** 65% of subway incidents have `Min Delay = 0` (logged, but no delay). A few huge values (up to 999) look like placeholders.
`Vehicle = 0` means "no vehicle" in a third of rows. 269 exact duplicate rows.

In [11]:
codes = pd.read_csv(RAW / "ttc-subway-delay-data/code-descriptions.csv")
print(codes.head(), "\n")
unknown = subway.loc[~subway["Code"].isin(codes["CODE"]), "Code"]
print(f"{len(unknown)} rows ({len(unknown) / len(subway):.1%}) use a code missing from code-descriptions.csv")
unknown.value_counts().head(15)

   _id   CODE          DESCRIPTION
0    1   EUAC     AIR CONDITIONING
1    2   EUAL  ALTERNATING CURRENT
2    3  EUATC   ATC RC&S EQUIPMENT
3    4   EUBK               BRAKES
4    5   EUBO                 BODY 

9832 rows (3.7%) use a code missing from code-descriptions.csv


Code
MRO      1432
SUL      1132
MRTO      900
PUTO      686
ERDO      674
ERTC      587
MRWEA     370
MRUI      285
MRPAA     234
SRDP      229
ERPR      220
SRO       202
MUINF     202
PREL      178
TRNCA     140
Name: count, dtype: int64

## 3. Bus

In [12]:
bus_parts = load_sheets("bus")
column_layouts(bus_parts)

 67 sheet(s): ['Report Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle']
      first: ttc-bus-delay-data-2014.xlsx [Jan 2014]
      last:  ttc-bus-delay-data-2019.xlsx [Oct 2019 ]

  1 sheet(s): ['Report Date', 'Route', 'Time', 'Day', 'Location', 'Incident', ' Min Delay', 'Min Gap', 'Direction', 'Vehicle']
      first: ttc-bus-delay-data-2018.xlsx [Mar 2018 ]
      last:  ttc-bus-delay-data-2018.xlsx [Mar 2018 ]

  1 sheet(s): ['Report Date', 'Route', 'Time', 'Day', 'Location', 'Incident ID', 'Incident', 'Delay', 'Gap', 'Direction', 'Vehicle']
      first: ttc-bus-delay-data-2019.xlsx [Apr 2019]
      last:  ttc-bus-delay-data-2019.xlsx [Apr 2019]

 10 sheet(s): ['Report Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Delay', 'Gap', 'Direction', 'Vehicle']
      first: ttc-bus-delay-data-2019.xlsx [June 2019]
      last:  ttc-bus-delay-data-2020.xlsx [July 2020]

 17 sheet(s): ['Date', 'Route', 'Time', 'Day', 'Location', 'Inc

**Finding:** bus changes layout constantly. The same column appears under different names:

| Meaning | Names used |
|---|---|
| date | `Report Date` (2014–2019), `Date` (2020+) |
| route | `Route`, `Line` (Jul–Aug 2021, CSV) |
| location | `Location`, `Station` (CSV) |
| reason | `Incident` (text), `Code` (CSV) |
| delay | `Min Delay`, `' Min Delay'` (leading space, Mar 2018), `Delay` (2019–2020) |
| gap | `Min Gap`, `Gap` |
| direction | `Direction`, `Bound` |

Plus stray columns: `Incident ID` (Apr 2019 only) and `Unnamed: 10` (Dec 2021, one value).

In [13]:
bus = pd.concat([p.rename(columns=RENAME) for p in bus_parts], ignore_index=True)
print(len(bus), "rows")
bus.groupby("_file")["date"].agg(["min", "max", "count"])

809655 rows


,min,max,count
_file,,,
ttc-bus-delay-data-2014.xlsx,2014-01-01 00:00:00,2014-12-31 00:00:00,94217
ttc-bus-delay-data-2015.xlsx,2015-01-01 00:00:00,2015-12-31 00:00:00,76510
ttc-bus-delay-data-2016.xlsx,2016-01-01 00:00:00,2016-12-31 00:00:00,77088
ttc-bus-delay-data-2017.xlsx,2017-01-01 00:00:00,2017-12-31 00:00:00,70303
ttc-bus-delay-data-2018.xlsx,2018-01-01 00:00:00,2018-12-31 00:00:00,73927
ttc-bus-delay-data-2019.xlsx,2019-01-01 00:00:00,2019-12-31 00:00:00,62376
ttc-bus-delay-data-2020.xlsx,2020-01-01 00:00:00,2020-12-31 00:00:00,36151
ttc-bus-delay-data-2021.xlsx,2021-01-01 00:00:00,2021-12-31 00:00:00,42269
ttc-bus-delay-data-2022.xlsx,2022-01-01 00:00:00,2022-12-31 00:00:00,58707


In [14]:
# Route: stored as int, float, or text depending on the file
bus["route_type"] = bus["route"].map(lambda v: type(v).__name__)
bus.groupby("_file")["route_type"].value_counts().unstack(fill_value=0)

route_type,float,int,str
_file,,,
ttc-bus-delay-data-2014.xlsx,0,94217,0
ttc-bus-delay-data-2015.xlsx,0,76510,0
ttc-bus-delay-data-2016.xlsx,0,77088,0
ttc-bus-delay-data-2017.xlsx,0,70303,0
ttc-bus-delay-data-2018.xlsx,0,73927,0
ttc-bus-delay-data-2019.xlsx,0,62376,0
ttc-bus-delay-data-2020.xlsx,11291,24860,0
ttc-bus-delay-data-2021.xlsx,11443,30805,21
ttc-bus-delay-data-2022.xlsx,384,58271,52


In [15]:
text_routes = bus["route"].astype(str)
text_routes[~text_routes.str.fullmatch(r"\d+(\.0)?")].value_counts().head(25)

route
52 LAWRENCE WEST         2387
32 EGLINTON WEST         2124
102 MARKHAM ROAD         1982
29 DUFFERIN              1907
7 BATHURST               1840
939 FINCH EXPRESS        1727
63 OSSINGTON             1669
54 LAWRENCE EAST         1618
116 MORNINGSIDE          1566
96 WILSON                1506
85 SHEPPARD EAST         1502
35 JANE                  1501
36 FINCH WEST            1500
86 SCARBOROUGH           1500
165 WESTON ROAD NORTH    1492
25 DON MILLS             1490
95 YORK MILLS            1476
53 STEELES EAST          1435
47 LANSDOWNE             1306
41 KEELE                 1231
24 VICTORIA PARK         1194
929 DUFFERIN EXPRESS     1163
43 KENNEDY               1139
89 WESTON                1127
37 ISLINGTON             1084
Name: count, dtype: int64

**Finding:** until 2019 `route` is a number (`52`). From 2020 it's sometimes a float (`52.0`), and in the CSV it's text with the name (`52 LAWRENCE WEST`, names also cut at 22 chars).

In [16]:
bus["bound"].value_counts(dropna=False).head(30)

bound
N      98065
S      89483
E      87743
W      81397
NaN    70401
N/B    44475
B/W    43696
S/B    40214
E/B    39272
W/B    36498
NB     14175
WB     13137
BW     13119
EB     13054
n      12885
s      11184
SB     11065
w      10472
e       9494
n/b     8563
s/b     8260
b/w     7303
e/b     5481
nb      4947
w/b     4874
eb      4119
sb      3607
bw      3586
wb      3574
B       2310
Name: count, dtype: int64

**Finding:** direction is written ~40 ways: `N`, `n`, `NB`, `N/B`, `n/b`, `nb`… and `B/W`, `BW`, `b/w` (= both ways).

In [17]:
is_csv = bus["_file"].str.endswith(".csv")
print("Incident, 2014–2024 files:")
print(bus.loc[~is_csv, "incident"].value_counts().head(15), "\n")
print("Incident, since-2025 CSV:")
print(bus.loc[is_csv, "incident"].value_counts().head(10))

Incident, 2014–2024 files:
incident
Mechanical                          247737
Utilized Off Route                   87023
General Delay                        79177
Late Leaving Garage                  65450
Operations - Operator                51266
Investigation                        50993
Diversion                            40409
Emergency Services                   21260
Security                             16765
Collision - TTC                      14836
Cleaning                             11031
Cleaning - Unsanitary                 6598
Vision                                5067
Road Blocked - NON-TTC Collision      2566
Late Leaving Garage - Mechanical      1865
Name: count, dtype: int64 

Incident, since-2025 CSV:
incident
EFO      18178
TFCNO     8742
MFSH      7730
MFDV      7288
TFPD      6776
TFO       5593
MFUS      4272
EFD       4078
SFDP      3747
MFSAN     2745
Name: count, dtype: int64


**Finding (important):** the *meaning* of the reason column changed in 2025.
Until 2024 it's a plain-English category (`Mechanical`, `Diversion`); in the CSV it's a TTC code (`EFO`, `MFDV`) that needs `code-descriptions.csv`.
These are two different systems, so they need two separate columns, not one.

In [18]:
print(bus["location"].nunique(), "distinct locations\n")
kennedy = bus["location"].astype(str).str.upper() == "KENNEDY STATION"
bus.loc[kennedy, "location"].value_counts()

137446 distinct locations



location
KENNEDY STATION    9489
Kennedy Station    4025
Kennedy station    1294
kennedy station     359
kennedy Station      43
Kennedy STAtion       1
KENNEDY Station       1
Name: count, dtype: int64

**Finding:** 137k distinct location strings. Casing alone gives 7 versions of `KENNEDY STATION`; there are also `Kennedy Stn`, `STC`, `Entire Route`…
Free-text locations can't be fully normalized. Plan: uppercase + trim + expand `STN` → `STATION`, then only map the common ones.

In [19]:
print("time value types:")
print(bus["time"].map(lambda v: type(v).__name__).value_counts(), "\n")
print(bus["min_delay"].describe(), "\n")
print("negative delays:", (bus["min_delay"] < 0).sum())
print("delays over 10 hours:", (bus["min_delay"] > 600).sum())
print("largest:", sorted(bus["min_delay"].dropna())[-6:])

time value types:


time
time        479248
str         330374
datetime        33
Name: count, dtype: int64 

count    809191.000000
mean         20.301701
std         281.159652
min         -54.000000
25%           7.000000
50%          10.000000
75%          17.000000
max      246245.000000
Name: min_delay, dtype: float64 

negative delays: 10
delays over 10 hours: 2744


largest: [2430.0, 3380.0, 3780.0, 5775.0, 6518.0, 246245.0]


**Finding:** `time` is a mix of Python `time` objects (Excel) and `"HH:MM"` strings, plus 33 full datetimes. `min_delay` has negatives and absurd values (246,245 min ≈ 171 days).

In [20]:
cols = ["date", "route", "time", "location", "incident", "min_delay", "min_gap", "bound", "vehicle"]
print("exact duplicate rows:", bus[cols].astype(str).duplicated().sum())
bus.isna().sum()

exact duplicate rows: 1788


date                0
route            3135
time                0
day                 0
location          793
incident          935
min_delay         464
min_gap           944
bound           70401
vehicle         69840
_file               0
_sheet              0
Incident ID    804511
Unnamed: 10    809654
_id            707398
route_type          0
dtype: int64

## 4. Streetcar

In [21]:
streetcar_parts = load_sheets("streetcar")
column_layouts(streetcar_parts)

 68 sheet(s): ['Report Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle']
      first: ttc-streetcar-delay-data-2014.xlsx [Jan 2014]
      last:  ttc-streetcar-delay-data-2019.xlsx [Oct 2019 ]

  1 sheet(s): ['Report Date', 'Route', 'Time', 'Day', 'Location', 'Incident ID', 'Incident', 'Delay', 'Gap', 'Direction', 'Vehicle']
      first: ttc-streetcar-delay-data-2019.xlsx [Apr 2019]
      last:  ttc-streetcar-delay-data-2019.xlsx [Apr 2019]

 17 sheet(s): ['Report Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Delay', 'Gap', 'Direction', 'Vehicle']
      first: ttc-streetcar-delay-data-2019.xlsx [June 2019 ]
      last:  ttc-streetcar-delay-data-202009.xlsx [July 2020]

 25 sheet(s): ['Date', 'Line', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle']
      first: ttc-streetcar-delay-data-2020.xlsx [Aug 20]
      last:  ttc-streetcar-delay-data-2024.xlsx [Data]

  1 sheet(s): ['_id', 'Date', 'Li

Same column-name churn as bus (`Report Date`/`Date`, `Route`/`Line`, `Min Delay`/`Delay`, `Direction`/`Bound`, `Incident ID`). Streetcar switches to `Line`/`Bound` from Aug 2020 on.

In [22]:
streetcar = pd.concat([p.rename(columns=RENAME) for p in streetcar_parts], ignore_index=True)
streetcar.groupby("_file")["date"].agg(["min", "max", "count"])

,min,max,count
_file,,,
ttc-streetcar-delay-data-2014.xlsx,2014-01-02 00:00:00,2014-12-31 00:00:00,11027
ttc-streetcar-delay-data-2015.xlsx,2015-01-01 00:00:00,2015-12-31 00:00:00,12221
ttc-streetcar-delay-data-2016.xlsx,2016-01-01 00:00:00,2016-12-31 00:00:00,14021
ttc-streetcar-delay-data-2017.xlsx,2017-01-01 00:00:00,2017-12-31 00:00:00,13762
ttc-streetcar-delay-data-2018.xlsx,2018-01-01 00:00:00,2018-12-31 00:00:00,15612
ttc-streetcar-delay-data-2019.xlsx,2019-01-01 00:00:00,2019-12-31 00:00:00,11882
ttc-streetcar-delay-data-2020.xlsx,2020-01-01 00:00:00,2020-12-31 00:00:00,7830
ttc-streetcar-delay-data-202009.xlsx,2020-01-01 00:00:00,2020-12-31 00:00:00,7830
ttc-streetcar-delay-data-2021.xlsx,2021-01-01 00:00:00,2021-12-31 00:00:00,14596


**Finding:** `ttc-streetcar-delay-data-2020.xlsx` and `…-202009.xlsx` cover exactly the same dates with exactly the same row count (see section 5).

In [23]:
text_routes = streetcar["route"].astype(str)
text_routes[~text_routes.str.fullmatch(r"\d+(\.0)?")].value_counts().head(30)

route
504 KING                  4102
501 QUEEN                 3992
505 DUNDAS                3230
506 CARLTON               2950
503 KINGSTON ROAD         1814
510 SPADINA               1761
512 ST. CLAIR             1616
511 BATHURST              1036
507 LONG BRANCH            805
509 HARBOURFRONT           616
301 QUEEN NIGHT            258
306 CARLTON NIGHT          167
304 KING NIGHT             145
305 DUNDAS NIGHT           136
508 LAKESHORE              103
RAD                         80
310 SPADINA NIGHT           55
305 DUNDAS                  32
312 ST. CLAIR NIGHT         30
303 KINGSTON ROAD NIGH      27
OTC                         14
600 RAD                     13
705 RAD                     12
303 KINGSTON ROAD            9
306 CARLTON                  8
301 QUEEN                    7
512 ST CLAIR                 7
504B KING                    6
303 KINGSTON RD              5
304 KING                     5
Name: count, dtype: int64

**Finding:** besides `504 KING`-style names: branch letters (`504A`, `504B`), night routes (`301 QUEEN NIGHT`), non-routes (`RAD`, `OTC`, `TEST CAR`), even subway lines (`BD`, `YU`).

In [24]:
print(streetcar["bound"].value_counts(dropna=False).head(25), "\n")
print(streetcar["incident"].value_counts().head(25))

bound
W/B          34169
E/B          33270
W            28367
E            28146
NaN          16762
S             9707
N             8950
N/B           6464
S/B           6376
B/W           5460
B              258
eb             178
EB             142
WB             118
wb             108
sb              27
bw              20
nb              20
NB              18
SB              18
EW              12
w/b              9
eastbound        8
ew               7
w                7
Name: count, dtype: int64 

incident
Mechanical                  48653
Operations                  20597
Held By                     14915
Investigation               13603
General Delay               12768
Emergency Services           8825
Late Leaving Garage          8155
Security                     6897
Diversion                    5370
Utilized Off Route           5213
Cleaning - Unsanitary        3859
Collision - TTC Involved     2666
MTDV                         2449
MTSAN                        2320
Cleani

Same direction mess as bus (plus `eastbound`, `EW`, digits). Incident text has a near-duplicate `Late ` with a trailing space, and junk `e`.

In [25]:
print(streetcar["location"].nunique(), "distinct locations")
print("exact duplicate rows:", streetcar[cols].astype(str).duplicated().sum())
print("largest delays:", sorted(streetcar["min_delay"].dropna())[-6:])
streetcar["location"].value_counts().head(25)

24969 distinct locations


exact duplicate rows: 8093
largest delays: [1185.0, 1294.0, 1294.0, 1294.0, 1294.0, 1400.0]


location
DUNDAS WEST STATION       2862
BROADVIEW STATION         1955
SPADINA STATION           1728
EXHIBITION LOOP           1227
QUEEN AND RONCESVALLES    1206
BATHURST STATION          1147
Russell Yard              1137
Queen and Connaught       1032
Leslie Barns               967
QUEEN AND LESLIE           967
LONG BRANCH LOOP           954
MAIN STREET STATION        945
HUMBER LOOP                938
Broadview Station          877
Roncesvalles and Queen     858
Dundas West Station        829
KING AND SUMACH            809
KINGSTON LOOP              802
Roncesvalles Yard          798
QUEEN AND BROADVIEW        792
Queen and Roncesvalles     788
QUEEN AND WOODFIELD        767
GERRARD AND BROADVIEW      760
BROADVIEW AND GERRARD      756
LESLIE BARNS               750
Name: count, dtype: int64

**Finding:** intersections are written in both orders and cases (`QUEEN AND RONCESVALLES`, `Roncesvalles and Queen`). The duplicate count is high mostly because of the double 2020 file.

## 5. Duplicate files

In [26]:
def md5(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

a = RAW / "ttc-streetcar-delay-data/ttc-streetcar-delay-data-2020.xlsx"
b = RAW / "ttc-streetcar-delay-data/ttc-streetcar-delay-data-202009.xlsx"
print(md5(a))
print(md5(b))
print("identical:", md5(a) == md5(b))

c1ce58324aa6189aefdf5133323525f9
c1ce58324aa6189aefdf5133323525f9
identical: True


Byte-for-byte identical → skip `202009` in the cleaner, or 2020 streetcar delays are counted twice.

## 6. Summary

| | Subway | Bus | Streetcar |
|---|---|---|---|
| Rows (raw) | 267,842 | 809,655 | 178,758 (7,830 are the duplicate file) |
| Column layouts | 2 | 8 | 5 |
| Distinct locations | 2,223 | 137,446 | 24,969 |
| Reason column | code (all years) | text → code in 2025 | text → code in 2025 |

Full list of issues and what to do about each: **[`../DATA_NOTES.md`](../DATA_NOTES.md)**.